In [1]:
!pip install pandas
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
import torch
print(torch.cuda.is_available())  # Should be True

False


In [1]:
#adding path of git 
import os

# Correct path string (with closing quotes)
os.environ["PATH"] += r";C:\Program Files\Git\cmd"

# Test if Git now works inside Jupyter
!git --version

git version 2.48.1.windows.1


In [2]:
cd starcoder2-main

C:\Users\adnan\Desktop\experiments\starcoder2-main


In [ ]:
!git --version

In [2]:
!pip install huggingface_hub


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from huggingface_hub import login

# Log in with your Hugging Face token
login(token="Api")

In [ ]:
#local Model prompting

In [23]:
from torch.utils.data import Dataset

class PuppetDataset(Dataset):
    def __init__(self, scripts):
        """
        Custom Dataset for Puppet script misconfiguration classification.
        
        Args:
            scripts (list): List of Puppet script texts.
        """
        self.encodings = tokenizer(
            scripts,
            padding=True,  # ✅ Ensure uniform length for batch processing
            truncation=True,
            max_length=512,
            return_tensors="pt"  # ✅ Converts directly to PyTorch tensors
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        """
        Returns a single example from the dataset.
        """
        return {
            "input_ids": self.encodings["input_ids"][idx], 
            "attention_mask": self.encodings["attention_mask"][idx]  # ✅ Include attention mask
        }



In [ ]:
tokenizer.pad_token = tokenizer.eos_token

# Loading CSV file
csv_file = "PuppetScripts_V2.csv"
df = pd.read_csv(csv_file)

# Ensure missing values are handled
df.dropna(subset=['Script.Content'], inplace=True)

# Convert to dataset
dataset = PuppetDataset(df['Script.Content'].tolist())

In [ ]:
from transformers import AutoTokenizer

# Loading tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-7b")

tokenizer.pad_token = tokenizer.eos_token  # ✅ Now padding will work


In [27]:
from torch.utils.data import Dataset

class PuppetDataset(Dataset):
    def __init__(self, scripts):
        """
        Custom Dataset for Puppet script misconfiguration classification.
        
        Args:
            scripts (list): List of Puppet script texts.
        """
        self.encodings = tokenizer(
            scripts,
            padding=True,  # ✅ Ensure uniform length for batch processing
            truncation=True,
            max_length=512,
            return_tensors="pt"  # ✅ Converts directly to PyTorch tensors
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        """
        Returns a single example from the dataset.
        """
        return {
            "input_ids": self.encodings["input_ids"][idx], 
            "attention_mask": self.encodings["attention_mask"][idx]  # ✅ Include attention mask
        }


In [28]:
from transformers import AutoModelForCausalLM

# Load model
model = AutoModelForCausalLM.from_pretrained("bigcode/starcoder2-7b")

# 🔥 Ensure model config has `pad_token_id`
model.config.pad_token_id = tokenizer.pad_token_id


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

C:\Users\adnan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adnan\.cache\huggingface\hub\models--bigcode--starcoder2-7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Define model paths
checkpoint = "bigcode/starcoder2-7b"
model_save_path = "./saved_starcoder_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Try loading from local storage, otherwise download and save
if os.path.exists(model_save_path):
    print("✅ Loading model from disk...")
    tokenizer = AutoTokenizer.from_pretrained(model_save_path)
    model = AutoModelForCausalLM.from_pretrained(model_save_path).to(device)
else:
    print("🔽 Downloading model from Hugging Face...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

    # Save model for future use
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"✅ Model saved locally at {model_save_path}")

print("✅ Model ready for use!")


✅ Loading model from disk...


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Model ready for use!


In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Define model paths
checkpoint = "bigcode/starcoder2-7b"
model_save_path = "./saved_starcoder_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Try loading from local storage, otherwise download and save
if os.path.exists(model_save_path):
    print("✅ Loading model from disk...")
    tokenizer = AutoTokenizer.from_pretrained(model_save_path)
    model = AutoModelForCausalLM.from_pretrained(model_save_path).to(device)
else:
    print("🔽 Downloading model from Hugging Face...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

    # Save model for future use
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"✅ Model saved locally at {model_save_path}")

print("✅ Model ready for use!")

C:\Users\adnan\Downloads\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loading model from disk...


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 10.93it/s]


✅ Model ready for use!


In [8]:
import pandas as pd
import json

# Load CSV file containing Puppet scripts
csv_file = "PuppetScripts_V2.csv"  # Change this to your actual CSV file
df = pd.read_csv(csv_file)

# Prepare lists to store results
classifications = []
reasons = []

# Process each script one by one
for index, row in df.iterrows():
    script_content = row["Script.Content"]
    
    # Analyze misconfiguration
    result = analyze_misconfiguration(script_content)

    # Store classification and reason
    classifications.append(result["classification"])
    reasons.append(result["reason"])

    # Print the result immediately
    print(f"✅ Processed Row {index + 1}/{len(df)}")
    print(f"Classification: {result['classification']}")
    print(f"Reason: {result['reason']}")
    print("-" * 50)  # Separator

# Add results to DataFrame
df["classification"] = classifications
df["reason"] = reasons

# Save results to a new CSV file
output_file = "misconfiguration_results.csv"
df.to_csv(output_file, index=False)

print(f"✅ Classification completed! Results saved to {output_file}")

C:\Users\adnan\Downloads\myenv\lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\adnan\Downloads\myenv\lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\adnan\Downloads\myenv\lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


KeyboardInterrupt: 

In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True if GPU is detected
print(torch.cuda.device_count())  # Should return the number of GPUs available
print(torch.cuda.get_device_name(0))  # Should return the GPU name
print(torch.cuda.current_device())  # Should return the current GPU index


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU detected:", torch.cuda.device_count())

Torch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU detected: 1


In [5]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Jan_15_19:38:46_Pacific_Standard_Time_2025
Cuda compilation tools, release 12.8, V12.8.61
Build cuda_12.8.r12.8/compiler.35404655_0


In [1]:
import sys
print(sys.executable)
print(sys.version)


C:\Users\adnan\AppData\Local\Programs\Python\Python310\python.exe
3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [1]:
import torch
print("PyTorch Version:", torch.__version__)


PyTorch Version: 2.5.1+cu121


In [13]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
#Prompt starcoder API

In [50]:
import os
import requests
import pandas as pd
import json
import re
from time import sleep

import json
import re

def extract_json(response_text):
    """
    Extracts the JSON portion from a long response text.
    If the response text is empty, it returns an error message.
    """
    
    # Check if response_text is empty or contains only spaces
    if not response_text.strip():
        return {"error": "Response text is empty."}

    # Use regex to find a JSON object (detects the first {...} block)
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)

    if json_match:
        json_str = json_match.group(0)  # Extract the JSON string
        try:
            return json.loads(json_str)  # Convert to a dictionary
        except json.JSONDecodeError:
            return {"error": "Invalid JSON format extracted."}

    return {"error": "No valid JSON found in response."}


# Load API Key from Environment Variable
API_KEY = "Api"  # Replace with your actual API key
if not API_KEY:
    raise ValueError("Missing Hugging Face API Key! Set 'HF_API_KEY' as an environment variable.")

API_URL = "https://api-inference.huggingface.co/models/bigcode/starcoder2-3b"
headers = {"Authorization": f"Bearer {API_KEY}"}

# Function to analyze misconfiguration

def analyze_misconfiguration(code_snippet):
   

    if not code_snippet.strip():
        return {"classification": -1, "reason": "Empty script.", "misconfigured_snippet": "N/A"}

    # Define structured prompt with clear instructions
    prompt = f"""
    You are an expert in Puppet configuration management and security. Analyze the following Puppet code for any misconfiguration, security issues.
    
    ### **Task**
    - Identify any **misconfigurations** and **security vulnerabilities**.
    - **Explain each issue clearly** and categorize its **severity** as Low, Medium, or High.
    - Provide a **corrected version of the code**.


      ### Security Issues to Check in the :
    - **Insecure file permissions**
    - **Hardcoded credentials**
    - **Insecure command execution**
    - **Missing security settings**
    - **Use of outdated or deprecated modules**
    - **Unrestricted network access**
    - **Misconfigured user privileges**
    - **Missing logging or monitoring**
    - **Admin by default**
    - **Empty password**
    - **Hard-coded secret**
    - **Invalid IP address binding**
    - **Suspicious comments**
    - **Use of HTTP without TLS**
    - **Use of weak cryptographic algorithms**
    - **Your Database for pupet**

      
    ```puppet code to anlyse
     {code_snippet}
    ```
    
    ### **Response Format**
    Return the response in **JSON format**:
    ```json
    {{
      "issues_found": [
        {{
          "type": "<Issue Type>",
          "description": "<Detailed Explanation of the Issue>",
          "severity": "<Low/Medium/High>",
          "misconfigured_snippet": "specific misconfigured code from the input script."
          "fixed_code": "<Provide a corrected version of the Puppet code>"
        }}
      ],
      
    }}
      ### Now, respond ONLY in JSON format  (without any additional text) as structured .
     """

    # Send request to Hugging Face API
    response = requests.post(
        API_URL,
        headers=headers,
        json={"inputs": prompt, "parameters": {"max_new_tokens":3000, "temperature": 1, "top_p": 0.8, "top_k": 10, "do_sample": False}}
    )

    # Handle API error cases
    if response.status_code != 200:
        return {
            "classification": -1,
            "reason": f"API Error: {response.status_code} - {response.text}",
            "misconfigured_snippet": "N/A"
        }

    try:
        # Extract raw API response
        result = response.json()
        generated_text = result[0]["generated_text"].strip()
        
        print(f"🔹 Raw API Response:\n{generated_text}\n")  # Debugging output

        # Try direct JSON parsing
        json_output = json.loads(generated_text)
        return json_output

    except (ValueError, json.JSONDecodeError):
        # Extract JSON using regex as a fallback
        json_match = re.search(r'\{.*?\}', generated_text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except json.JSONDecodeError:
                pass  # Ignore and move to heuristic fallback

    # Final fallback: Heuristic classification based on response
    if "No misconfiguration" in generated_text:
        return {
            "classification": 0,
            "reason": "No security misconfigurations detected.",
            "misconfigured_snippet": "N/A"
        }
    elif "Misconfiguration" in generated_text or "Insecure" in generated_text:
        return {
            "classification": 1,
            "reason": generated_text,
            "misconfigured_snippet": "Could not extract misconfiguration."
        }
    else:
        return {
            "classification": -1,
            "reason": "Unclear response from model.",
            "misconfigured_snippet": "N/A"
        }


  
# Load CSV file
csv_file = "PuppetScripts_V2.csv"
df = pd.read_csv(csv_file)

# Define output CSV file
output_file = "misconfiguration_results.csv"

# Create the CSV file with headers (overwrite if exists)
df_output = pd.DataFrame(columns=["Script.Content", "Classification", "Reason", "Misconfigured Snippet"])
df_output.to_csv(output_file, index=False, encoding="utf-8-sig", quoting=1)

# Process each row and save immediately
for index, row in df.iterrows():
    script_content = row.get("Script.Content", "").strip()

    if not script_content:
        print(f"⚠️ Skipping Row {index + 1} (Empty Script)")
        continue

    # Analyze misconfiguration
    result = analyze_misconfiguration(script_content)

    # Create a temporary DataFrame for the single row
    df_temp = pd.DataFrame({
        "Script.Content": [script_content],
        "Classification": [result.get("classification", -1)],
        "Reason": [result.get("reason", "No reason provided")],
        "Misconfigured Snippet": [result.get("misconfigured_snippet", "No snippet extracted")]
    })

    # Append to CSV immediately (mode='a' for append)
    df_temp.to_csv(output_file, mode='a', index=False, header=False, encoding="utf-8-sig", quoting=1)

    # Print real-time progress
    print(f"✅ Processed Row {index + 1}/{len(df)}: Classification {result.get('classification')}")
    sleep(0.5)  # Optional: Add a delay to prevent API rate limits

print(f"✅ Classification completed! Results saved to {output_file}")




🔹 Raw API Response:
You are an expert in Puppet configuration management and security. Analyze the following Puppet code for any misconfiguration, security issues.
    
    ### **Task**
    - Identify any **misconfigurations** and **security vulnerabilities**.
    - **Explain each issue clearly** and categorize its **severity** as Low, Medium, or High.
    - Provide a **corrected version of the code**.


      ### Security Issues to Check in the :
    - **Insecure file permissions**
    - **Hardcoded credentials**
    - **Insecure command execution**
    - **Missing security settings**
    - **Use of outdated or deprecated modules**
    - **Unrestricted network access**
    - **Misconfigured user privileges**
    - **Missing logging or monitoring**
    - **Admin by default**
    - **Empty password**
    - **Hard-coded secret**
    - **Invalid IP address binding**
    - **Suspicious comments**
    - **Use of HTTP without TLS**
    - **Use of weak cryptographic algorithms**
    - **Your 

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
#Api prompting StarCoder2

In [ ]:
import pandas as pd
import json
import requests
from time import sleep
import re

# Function to analyze misconfiguration (calls the Hugging Face API)
def analyze_misconfiguration(code_snippet):
    if not code_snippet.strip():
        return {"classification": -1, "reason": "Empty script.", "misconfigured_snippet": "N/A"}

    # Define structured prompt with clear instructions
    prompt = f"""
    You are an expert in Puppet configuration management and security. Analyze the following Puppet code for any misconfiguration, security issues, and best practices.

    ### **Task**
    - Identify any **misconfigurations** and **security vulnerabilities**.




    ### Security Issues to Check in the:
    - **Insecure file permissions**
    - **Hardcoded credentials**
    - **Insecure command execution**
    - **Missing security settings**
    - **Use of outdated or deprecated modules**
    - **Unrestricted network access**
    - **Misconfigured user privileges**
    - **Missing logging or monitoring**
    - **Admin by default**
    - **Empty password**
    - **Hard-coded secret**
    - **Invalid IP address binding**
    - **Suspicious comments**
    - **Use of HTTP without TLS**
    - **Use of weak cryptographic algorithms**
    
    ### Puppet Script to Review and make security analysis based on misconfigurations listed before:
    ```puppet
    {code_snippet}
    ```

    ### **Response Format**
    Return the response in **JSON format**:
    ```json
    {{
      "issues_found": [
        {{
          "type": "<Issue Type>",
          "description": "<Detailed Explanation of the Issue>",
          "severity": "<Low/Medium/High>",
          "recommended_fix": "<Suggested Fix>"
        }}
      ],
      "fixed_code": "<Provide a corrected version of the Puppet code>"
    }}
    ```
    ### Now, respond ONLY in JSON format  (without any additional text) as structured.
    """

    # API Request to Hugging Face
    API_KEY = "Api"  # Replace with your actual API key
    API_URL = "https://api-inference.huggingface.co/models/bigcode/starcoder2-3b"
    headers = {"Authorization": f"Bearer {API_KEY}"}

    response = requests.post(
        API_URL,
        headers=headers,
        json={"inputs": prompt, "parameters": {"max_new_tokens": 1000, "temperature": 1, "top_p": 0.8, "top_k": 10, "do_sample": False}}
    )

    if response.status_code != 200:
        return {"classification": -1, "reason": f"API Error: {response.status_code} - {response.text}", "misconfigured_snippet": "N/A"}

    try:
        result = response.json()
        generated_text = result[0]["generated_text"].strip()
        
        # Debug: print the raw text response
        print(f"🔹 Raw API Response:\n{generated_text}\n")

        # Search for response_marker and json_marker in the generated text
        response_marker = "### **Response Example**"
        json_marker = "```json"

        # Find the content after the response_marker and json_marker
        response_start = generated_text.find(response_marker)
        json_start = generated_text.find(json_marker, response_start)

        if json_start == -1:
            print("⚠️ No JSON block found in the response.")
            return {"classification": -1, "reason": "No valid JSON found in response.", "misconfigured_snippet": "N/A"}

        # Extract everything after the json_marker
        json_str = generated_text[json_start + len(json_marker):].strip()

        # Remove the code block closing "```"
        json_end = json_str.rfind("```")
        if json_end != -1:
            json_str = json_str[:json_end].strip()

        # Parse the extracted JSON part
        try:
            json_output = json.loads(json_str)
            return json_output
        except json.JSONDecodeError:
            return {"classification": -1, "reason": "Invalid JSON format extracted.", "misconfigured_snippet": "N/A"}

    except (ValueError, json.JSONDecodeError):
        return {"classification": -1, "reason": "Error processing response.", "misconfigured_snippet": "N/A"}


# Load the CSV file
csv_file = "PuppetScripts_V2.csv"
df = pd.read_csv(csv_file)

# Define output CSV file
output_file = "misconfiguration_results.csv"

# Create the CSV file with headers (overwrite if exists)
df_output = pd.DataFrame(columns=["Script.Content", "Classification", "Reason", "Misconfigured Snippet", "Fixed Code"])
df_output.to_csv(output_file, index=False, encoding="utf-8-sig", quoting=1)

# Process each row and save immediately
for index, row in df.iterrows():
    # Ensure that "Script.Content" is treated as a string
    script_content = str(row.get("Script.Content", "")).strip()

    if not script_content:
        print(f"⚠️ Skipping Row {index + 1} (Empty Script)")
        continue

    # Analyze misconfiguration using the Hugging Face model
    result = analyze_misconfiguration(script_content)

    # Extract information from the API response
    classification = result.get("classification", "No classification provided")
    reason = result.get("reason", "No reason provided")
    misconfigured_snippet = result.get("misconfigured_snippet", "No snippet extracted")
    fixed_code = result.get("fixed_code", "No fixed code provided")

    # Create a temporary DataFrame for the single row
    df_temp = pd.DataFrame({
        "Script.Content": [script_content],
        #"Classification": [classification],
        #"Reason": [reason],
        "Misconfigured Snippet": [misconfigured_snippet],
        #"Fixed Code": [fixed_code]
    })

    # Append to CSV immediately (mode='a' for append)
    df_temp.to_csv(output_file, mode='a', index=False, header=False, encoding="utf-8-sig", quoting=1)

    # Print real-time progress
    print(f"✅ Processed Row {index + 1}/{len(df)}: Classification {classification}")
    sleep(0.5)  # Optional: Add a delay to prevent API rate limits

print(f"✅ Classification completed! Results saved to {output_file}")

🔹 Raw API Response:
You are an expert in Puppet configuration management and security. Analyze the following Puppet code for any misconfiguration, security issues, and best practices.

    ### **Task**
    - Identify any **misconfigurations** and **security vulnerabilities**.
    - Evaluate the code against **best practices**.
    - **Explain each issue clearly** and categorize its **severity** as Low, Medium, or High.
    - Suggest a **recommended fix** for each issue.
    - Provide a **corrected version of the code**.

    ### Security Issues to Check in the:
    - **Insecure file permissions**
    - **Hardcoded credentials**
    - **Insecure command execution**
    - **Missing security settings**
    - **Use of outdated or deprecated modules**
    - **Unrestricted network access**
    - **Misconfigured user privileges**
    - **Missing logging or monitoring**
    - **Admin by default**
    - **Empty password**
    - **Hard-coded secret**
    - **Invalid IP address binding**
    - **

KeyboardInterrupt: 